## Tests — UI / analytics helpers

Unit tests for the deterministic MCC mapping used by the backend (`model/analytics_core.py`) and exposed to the UI.

These tests do not start Streamlit; they only validate helper behavior.

## Imports

In [4]:
from __future__ import annotations

import importlib.util
import pandas as pd
import plotly.express as px
from pathlib import Path


## Load backend helpers from `model/analytics_core.py` (+ UI client-id parser)

In [5]:
import sys


def find_project_root(start: Path | None = None) -> Path:
    """Find project root by locating the `data/` directory."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing a 'data/' directory")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model.analytics_core import mcc_to_category

app_path = ROOT / "ui" / "app.py"
spec = importlib.util.spec_from_file_location("ui_app", app_path)
assert spec and spec.loader
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
parse_client_ids = mod.parse_client_ids
format_usd_yaxis = mod.format_usd_yaxis
compute_yearly_category_trend = mod.compute_yearly_category_trend


## Unit tests

In [6]:
assert mcc_to_category("5812", "Eating Places and Restaurants") == "Dining"
assert mcc_to_category("5411", "Grocery Stores, Supermarkets") == "Groceries"
assert mcc_to_category("4900", "Utilities - Electric, Gas, Water, Sanitary") == "Utilities"
assert mcc_to_category("5814", "") == "Dining"  # prefix fallback
assert mcc_to_category(None, None) == "Other/Uncategorized"

ids = parse_client_ids(pd.DataFrame({"id": ["10", "2", None, "bad", "10"]}))
assert ids == [2, 10]

fig = px.bar(pd.DataFrame([{"x": "a", "spend_usd": 240000}]), x="x", y="spend_usd")
format_usd_yaxis(fig)
assert fig.layout.yaxis.tickprefix == "$"
assert fig.layout.yaxis.tickformat == ",.0f"

trend = compute_yearly_category_trend(
    pd.DataFrame(
        {
            "transaction_dt": ["2010-01-01", "2010-02-01", "2011-01-01", "2011-02-01"],
            "category": ["Dining", "Dining", "Groceries", "Dining"],
            "amount_usd": [10, 20, 5, -7],
        }
    ),
    top_n=2,
)
assert set(trend.columns) == {"year", "category", "spend_usd"}
assert trend["year"].min() == 2010
assert trend["year"].max() == 2011
assert (trend["spend_usd"] >= 0).all()

print("UI helper tests: PASS")


UI helper tests: PASS
